# LLM 评测方法论：系统更快以后，模型还是“同一个模型”吗？

> 前面已经把现代推理系统拆到了 Scheduler、KV Cache、Quantization、Speculative Decoding、PD 分离。
>
> 但优化系统不能只看速度。
>
> 例如：
>
> ```text
> BF16 → INT4
> Throughput +70%
> Memory -75%
> ```
>
> 如果模型准确率掉了 10%，这个优化未必值得。
>
> 本章只回答一个问题：**怎么公平、可复现地判断“质量有没有变、系统有没有真的变好”？**
>
> 我们不从工具列表开始，而从一条完整评测流水线开始：
>
> ```text
> Question / Dataset
> ↓
> Prompt Protocol
> ↓
> Generation Config
> ↓
> Model Output
> ↓
> Parser
> ↓
> Metric / Judge
> ↓
> Aggregation / Confidence Interval
> ```
>
> 读完以后，希望你看到 `MMLU / GSM8K / HumanEval / IFEval / LLM-as-Judge / pass@k / win-rate / bootstrap / lm-eval` 时，知道它们分别处在哪一层。

先从一个很小的问题开始。

## 1. 14/20 和 15/20，真的能说 B 更好吗？

模型 A：

```text
20 道题答对 14 道
```

模型 B：

```text
20 道题答对 15 道
```

单看分数：

```text
A = 70%
B = 75%
```

很容易说 B 更好。

但如果换一批题，顺序可能反转。

所以评测首先要接受一件事：

> **一个分数也是一个统计估计，不是真理。**

In [ ]:
import math

def accuracy(correct, total):
    p = correct / total
    se = math.sqrt(p * (1-p) / total)
    return p, se

for name, correct in [("A", 14), ("B", 15)]:
    p, se = accuracy(correct, 20)
    print(f"{name}: acc={p:.1%}, rough SE={se:.3f}")

20 道题太少，随机波动很大。

这就是为什么后面会讨论：

- 数据集规模
- Bootstrap
- Confidence Interval
- Paired Comparison

但在统计之前，还有更基础的问题：

> **两次评测真的用了完全相同的 Prompt、Sampling 和 Parser 吗？**

## 2. Benchmark 不等于“一组题”

一个可复现 Benchmark 至少包含：

```text
数据本身
+ prompt format
+ few-shot examples
+ answer extraction
+ metric
```

例如同一道人类看来完全一样的选择题：

```text
答案是 A/B/C/D
```

可以有很多 Prompt：

```text
Answer:
```

或者：

```text
The correct answer is:
```

或者 Chat Template：

```text
system + user + assistant
```

不同 Prompt 可能导致不同分数。

所以论文或模型报告里的：

```text
MMLU 82.3
```

必须结合评测协议看。

## 3. Generation Config：Temperature 也会改变 Benchmark

上一章已经学过：

- temperature
- top-p
- max_tokens
- repetition penalty
- stop token

这些不是“服务参数而已”。

如果 Benchmark 是生成式任务，它们会直接改变结果。

例如数学题：

```text
temperature=0
```

通常更稳定。

创意对话：

```text
temperature=0.7
```

可能更合理。

所以公平比较至少要固定：

```text
model version
prompt
chat template
temperature
top_p
max_tokens
seed（如果框架支持）
stop conditions
```

这就是 **Evaluation Protocol**。

## 4. Parser：模型答对了，但解析器说错了？

假设标准答案是：

```text
42
```

模型输出：

```text
经过计算，最终答案是 42。
```

如果评测器直接做：

```python
output == "42"
```

就会判错。

所以生成式评测通常需要 **Answer Extraction / Parser**。

Parser 本身也会影响分数。

In [ ]:
outputs = [
    "42",
    "The answer is 42.",
    "最终答案：42",
    "I think it is 41.",
]

import re

def extract_last_integer(text):
    nums = re.findall(r"-?\d+", text)
    return nums[-1] if nums else None

for out in outputs:
    print(f"{out:<24} -> parsed={extract_last_integer(out)}")

所以看到两个 leaderboard 分数不一样，不要第一反应就是：

> “模型版本变了。”

也可能是：

- Prompt 变了。
- few-shot 变了。
- Parser 变了。
- Generation Config 变了。
- Dataset revision 变了。

这就是为什么成熟评测要把整个 Pipeline 固定。

## 5. Metric：不同任务到底怎么判分？

不同任务需要不同 Metric。

### Exact Match

答案必须完全一致。

适合：

- 数学最终答案
- 短问答
- 部分结构化任务

### Accuracy

选择题最常见。

### F1

常用于答案集合 / span overlap。

### pass@k

代码任务里常见。

生成多个候选，只要其中至少一个通过测试，就算成功。

### Success Rate

Agent / Tool-use 任务常见。

例如：

```text
SWE-bench
WebArena
BrowserGym
```

最终看任务是否完成。

In [ ]:
# pass@k 的一个直觉例子
# 10 个独立候选，每个成功概率 0.2
p_success = 0.2

for k in [1, 2, 5, 10]:
    pass_k = 1 - (1 - p_success) ** k
    print(f"k={k:2d} -> 理想化 pass@k={pass_k:.2%}")

真实 HumanEval / code benchmark 的 pass@k 有更严格的无偏估计方式，但这个小例子先建立直觉：

> **pass@1 和 pass@10 不是同一个能力指标。**

如果公司技术报告只把自己的 pass@10 和别人的 pass@1 放一起，就不能直接比较。

## 6. Perplexity：为什么它不是“模型总分”？

Perplexity 来自语言模型对 Token 的概率。

直觉上：

```text
模型越确信正确下一个 Token
→ loss 越低
→ perplexity 越低
```

它适合衡量语言建模能力、做量化前后 sanity check。

但它不直接等价于：

- 指令遵循
- 数学推理
- 对话质量
- Agent 成功率

所以不能用一个 PPL 代替所有下游评测。

In [ ]:
import math

for loss in [1.0, 1.5, 2.0, 3.0]:
    print(f"cross entropy={loss:.1f} -> perplexity={math.exp(loss):.2f}")

## 7. LLM-as-Judge：没有标准答案怎么办？

开放式问答很难 Exact Match。

例如：

```text
“解释为什么 KV Cache 能加速 Decode。”
```

不同表达都可能正确。

因此可以让另一个强模型打分：

```text
Candidate A
Candidate B
↓
Judge Model
↓
A wins / B wins / tie
```

这就是 **LLM-as-Judge**。

常见指标：

- score
- win rate
- pairwise preference
- Elo-like rating

但 Judge 也有偏差。

常见问题：

### Position Bias
A/B 顺序不同可能影响判断。

### Length Bias
更长的回答可能显得“更完整”。

### Style Bias
Judge 可能偏好特定格式和语气。

### Self-preference
Judge 可能更喜欢和自己风格相近的输出。

所以更严谨的做法包括：

```text
交换 A/B 顺序
长度控制
多 Judge
人工抽查
明确 Rubric
```

看到厂商写：

```text
GPT-4 Judge win-rate = 68%
```

至少继续问 Judge Prompt 和 Baseline 是什么。

## 8. Bootstrap：分数差多少才算“真的有差”？

如果我们有每道题的 0/1 结果，可以反复从题目中有放回抽样，重新算 Accuracy。

这就是 Bootstrap 的直觉。

我们用 200 道 toy result 做一次。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

# 假设模型 A/B 在同一批 200 题上的结果
a = rng.binomial(1, 0.70, size=200)
b = rng.binomial(1, 0.73, size=200)

diffs = []
for _ in range(5000):
    idx = rng.integers(0, len(a), size=len(a))
    diffs.append(b[idx].mean() - a[idx].mean())

lo, hi = np.percentile(diffs, [2.5, 97.5])
print("A acc:", a.mean())
print("B acc:", b.mean())
print("B-A 95% bootstrap interval:", (lo, hi))

如果区间跨过 0：

```text
[-0.04, +0.09]
```

就说明当前样本下还很难确信 B 稳定优于 A。

这比一句：

```text
B 高 3 分
```

信息丰富得多。

## 9. 为什么量化评测必须“质量 + 性能”一起看？

回到 Part 3 的主线。

假设：

| 方案 | GPU Memory | Throughput | MMLU |
|---|---:|---:|---:|
| BF16 | 16 GB | 100 tok/s | 70.0 |
| W8A16 | 9 GB | 130 tok/s | 69.9 |
| W4A16 | 5 GB | 180 tok/s | 68.5 |

没有一个单独指标能决定“最好”。

真正的工程问题是：

> 在你的质量预算和 SLO 下，哪个点是最好的 Pareto Point？

所以推理评测必须同时看：

```text
Quality
+ TTFT
+ TPOT
+ Throughput
+ Memory
+ Cost
```

这叫 **Quality-Performance Trade-off**。

In [ ]:
configs = [
    {"name":"BF16", "quality":70.0, "throughput":100, "memory":16},
    {"name":"W8A16", "quality":69.9, "throughput":130, "memory":9},
    {"name":"W4A16", "quality":68.5, "throughput":180, "memory":5},
]

quality_floor = 69.0
valid = [x for x in configs if x["quality"] >= quality_floor]
best = max(valid, key=lambda x: x["throughput"])

print("质量底线:", quality_floor)
print("满足质量底线的最高吞吐方案:", best["name"])

## 10. Serving Benchmark：为什么固定 Prompt Length 还不够？

系统性能高度依赖 workload。

至少要考虑：

```text
Input length distribution
Output length distribution
Concurrency
Arrival rate
Prefix sharing
Sampling config
SLO
```

例如：

```text
128 input / 128 output
```

和：

```text
32k input / 32 output
```

完全是两种系统负载。

前者 Decode 比重高。

后者 Prefill 比重高。

所以读 Serving Benchmark 时，Workload Definition 和数字本身同样重要。

## 11. 常见 Benchmark 名字放回地图

### Knowledge / Reasoning
- MMLU / MMLU-Pro
- GPQA
- GSM8K / MATH / AIME

### Code
- HumanEval / HumanEval+
- LiveCodeBench
- SWE-bench

### Instruction Following
- IFEval

### Dialogue / Preference
- MT-Bench
- AlpacaEval
- Arena-style pairwise

### Long Context
- RULER
- Needle-like retrieval

### Agent
- SWE-bench
- WebArena 等

不要试图把所有 Benchmark 背下来。

先问：

> **它想测哪一种能力？Metric 是什么？Pipeline 怎么跑？**

## 12. lm-evaluation-harness 放在哪一层？

`lm-evaluation-harness` 之类工具的价值是：

> 把大量 Dataset、Prompt、Metric、Model Adapter 组织成统一 Evaluation Pipeline。

所以它不是“评测理论本身”。

你应该先理解：

```text
Dataset
→ Prompt
→ Model
→ Parser
→ Metric
```

再去使用框架。

这样即使未来工具换了，你仍然知道框架在帮你自动化哪一步。

## 13. 一个最小可复现评测记录应该保存什么？

建议至少保存：

```text
model name / commit
tokenizer
quantization config
engine version
prompt template
generation config
dataset revision
metric implementation
random seed
hardware
batch / concurrency
input/output length
timestamp
```

这样几个月后重跑，才知道“为什么分数变了”。

## 14. 从评测自然走向部署

到这里，我们已经可以做一个完整决策：

```text
候选模型
↓
质量 Benchmark
↓
量化方案
↓
Serving Benchmark
↓
质量 / 性能 / 显存折中
↓
选最终部署方案
```

下一章就不再讲抽象系统原理，而是亲手完成：

> **Checkpoint → vLLM / SGLang → OpenAI-Compatible API → Streaming → Benchmark → Troubleshooting → Multi-GPU / PD Deployment**

## 小结

一次可信评测需要固定：

- **Dataset / Benchmark**
- **Prompt Protocol**
- **Generation Config**
- **Parser**
- **Metric / Judge**
- **Aggregation / Statistical Test**

工程上还要再加：

- TTFT
- TPOT / ITL
- Throughput
- GPU Memory
- Cost

不要问“哪个模型分最高”，而是问：

> **在我的 Workload 和 SLO 下，哪个方案的质量-性能折中最好？**

## 作业

1. 给同一组 100 道题构造两个模型的 0/1 结果，做 Bootstrap 差值区间。
2. 写一个简单 Parser，从“最终答案是 42”中提取 42。
3. 解释为什么 pass@1 不能和 pass@10 直接比较。
4. 设计一个比较 BF16、FP8、AWQ 的评测表，至少包含 6 个指标。
5. 找一个任意模型技术报告，记录它是否公开了 Prompt、Generation Config、Judge 和 Dataset Revision。